> 在线讲义：[第 8 章 · 综合案例：把一个 π0 级 VLA 装上机器人](https://wikieden.github.io/robot-edge-inference/lessons/08-case-study.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 8 章 · 综合案例：把一个 π0 级 VLA 装上机器人

配套讲义：仓库里 `ai-infra-edge/notes/lessons/08-case-study.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（π0 的结构来自论文；利用率 40%、816 个 token、Thor FP8 稠密按 FP4 稀疏的 1/4、预留 3 GB 是假设）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 容量 | $6.6 \le$ 总 $- 3$ | `plan(name)["fits"]` | 三块都放得下 |
| 第二步 AGX Orin 前缀 | $4.9\text{e}12 \div (138\text{e}12 \times 0.4)$ | `prefix_ms(138)` | 88.7 ms |
| 第二步 AGX Orin 动作专家 | $10 \times 0.6 \div 204.8$ | `action_ms(204.8)` | 29.3 ms |
| 第二步 Thor 合计 | 两段相加 | `plan("Jetson T5000（Thor）")["total"]` | 45.6 ms |
| 第三步 余量 | $500 \div$ 下限 | `plan(name)["margin"]` | 2.3 / 4.2 / 11.0 倍 |

π0 的 3.3B、300M 动作专家、10 步 flow、H = 50、50 Hz 来自论文；带宽、算力来自官方页面。
利用率 40%、前缀 816 个 token、Thor FP8 稠密按 FP4 稀疏的 1/4 算、系统预留 3 GB 都是编的假设。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
DEVICES = {  # 名字: (内存 GB, 带宽 GB/s, 稠密算力 万亿次/秒（INT8 或 FP8）)
    "Orin NX 16GB（Super）": (16, 102.4, 78),
    "AGX Orin 64GB": (64, 204.8, 138),
    "Jetson T5000（Thor）": (128, 273, 2070 / 4),
}

**函数 `prefix_ms`**：前缀算力下限：2 × 参数 × token ÷ (稠密算力 × 利用率)。

In [ ]:
def prefix_ms(dense_tops: float, params: float = 3e9, tokens: int = 816, util: float = 0.4) -> float:
    """前缀算力下限：2 × 参数 × token ÷ (稠密算力 × 利用率)。"""
    return 2 * params * tokens / (dense_tops * 1e12 * util) * 1000

**函数 `action_ms`**：动作专家带宽下限：每个 flow 步都把 300M 参数（bf16 0.6 GB）读一遍。

In [ ]:
def action_ms(bw_gbps: float, expert_gb: float = 0.6, steps: int = 10) -> float:
    """动作专家带宽下限：每个 flow 步都把 300M 参数（bf16 0.6 GB）读一遍。"""
    return steps * expert_gb / bw_gbps * 1000

**函数 `plan`**：把一台设备的容量、下限、预算一起算出来。

In [ ]:
def plan(name: str, model_gb: float = 6.6, reserve_gb: float = 3.0, budget_ms: float = 500) -> dict:
    """把一台设备的容量、下限、预算一起算出来。"""
    mem, bw, tops = DEVICES[name]
    p, a = prefix_ms(tops), action_ms(bw)
    return {"fits": model_gb <= mem - reserve_gb, "prefix": p, "action": a, "total": p + a,
            "margin": budget_ms / (p + a)}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：容量（π0 3.3B × 2 字节 = 6.6 GB，系统预留 3 GB 是编的）")
for name in DEVICES:
    print(f"  {name:<20} {'放得下' if plan(name)['fits'] else '放不下'}")

In [ ]:
print("\n第二步：两段下限（前缀算力受限 + 动作专家带宽受限）")
for name in DEVICES:
    r = plan(name)
    print(f"  {name:<20} 前缀 {r['prefix']:6.1f} ms + 动作 {r['action']:5.1f} ms = {r['total']:6.1f} ms")

In [ ]:
print("\n第三步：和 500 ms 预算比（50 Hz、每执行 25 步推理一次）")
for name in DEVICES:
    r = plan(name)
    print(f"  {name:<20} 预算 ÷ 下限 = {r['margin']:.1f} 倍余量")
print("  下限只是物理极限。GR00T 2 在 Thor 上，没优化的 PyTorch 比优化后慢 5.8 倍：")
print("  余量不到 5 倍的设备，先做量化、TensorRT（第 4、5 章）再谈上车")

## 3. 代码里才看得到的

- 前缀里没算 3 路图像编码器（SigLIP）单独的读写，也没算动作专家对前缀 KV 的读取；都是偏乐观的地方，真机上会更慢。
- Thor 的 FP8 稠密算力是按"FP4 稀疏 ÷ 4"推的，官方产品页没直接给这个数；换算方式写在 `DEVICES` 的注释里，查到数据表后要替换。

## 4. 自己改一改

- 把 `util` 改成 0.2，看哪块板子的余量先跌破 2 倍。
- 把 `tokens` 改成 2 路相机（560 个），看前缀能省多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先定预算再定容量，三块板子都放下；
> 前缀按算力，动作按带宽，加起来是下限；
> 预算除下限看余量，不到五倍先优化；
> 不改输出的先做，超时断网要回滚。